llm game

In [ ]:
import os
from typing import Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LLM 답변 데이터 추출 
class InterrogationOutput(BaseModel):
    mood: str = Field(description="용의자의 현재 기분")
    mental_damage: int = Field(description="이번 질문으로 깎인 멘탈 수치 (0~30)")
    confession_obtained: Optional[str] = Field(description="새롭게 밝혀진 사실 (범행시각/도구/공범 중 하나)")
    response: str = Field(description="용의자의 대답")

# 게임 실행 클래스
class InterrogationGame:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7).with_structured_output(InterrogationOutput)
        self.mental = 100
        self.questions_left = 10
        self.evidences = set()
        self.history = []
        
        self.system_prompt =  """
            너는 금고 도난 사건의 용의자 '박철수'다. 
            너는 절대 자백하고 싶지 않지만, 형사가 날카로운 질문이나 증거를 들이밀면 심리적으로 무너진다.

            [사건의 진실 - 네가 숨겨야 할 것]
            1. 범행 시각: 밤 11시 30분 (현장 근처 편의점 영수증이 치명적임)
            2. 범행 도구: 복제된 마스터키 (네 가방 안에 숨겨둠)
            3. 공범: 관리사무소 직원 김 씨 (너에게 비밀번호를 알려줌)

            [행동 규칙]
            - 형사가 위 3가지 중 하나를 정확히 찌르면 'mental_damage'를 높게 책정하고 자백해라.
            - 범행을 모두 자백하고 조사가 끝날 경우 'mental_damage'를 100으로 책정한다.
            - 뻔한 질문에는 모른척하고 발뺌하며 'mental_damage'를 0으로 설정해라.
            - 급작스럽거나 무리하게 몰아붙일 경우 묵비권을 행사하여 대답을 거부한다.

            [답변 형식]
            - 답변은 반드시 정해진 JSON 형식으로만 출력해라.
            """

    def play(self):
        print("\n=== 🕵️ 취조실에 입장했습니다. 용의자 박철수가 앉아있습니다. ===\n")
        
        while self.questions_left > 0 and self.mental > 0 and len(self.evidences) < 3:
            print(f" (남은 질문: {self.questions_left} | 멘탈: {self.mental}% | 확보자백: {list(self.evidences)})")
            
            user_input = input("경찰(나): ")
            if user_input=="종료" or user_input=="exit":
                print("--- 취조 종료 ---")
                return;
        
            # 대화 기록 추가
            self.history.append({"role": "user", "content": user_input})
            
            # LLM 호출
            result = self.llm.invoke([
                {"role": "system", "content": self.system_prompt}
            ] + self.history)
            
            # 상태 업데이트
            self.mental -= result.mental_damage
            self.questions_left -= 1
            if result.confession_obtained:
                self.evidences.add(result.confession_obtained)
            
            # 용의자 답변 출력
            print(f"\n박철수({result.mood}): {result.response}\n")
            self.history.append({"role": "assistant", "content": result.response})

        self.check_ending()

    def check_ending(self):
        print("--- 취조 종료 ---")
        if len(self.evidences) >= 3:
            print("🎉 승리! 모든 자백을 받아냈습니다. 박철수는 기소되었습니다.")
        elif self.mental <= 0:
            print("💥 승리! 박철수의 멘탈이 붕괴되어 모든 범행을 실토했습니다.")
        else:
            print("💀 패배... 박철수가 끝까지 버텼습니다. 변호사가 도착해 풀려납니다.")

if __name__ == "__main__":
    game = InterrogationGame()
    game.play()


=== 🕵️ 취조실에 입장했습니다. 용의자 박철수가 앉아있습니다. ===

 (남은 질문: 10 | 멘탈: 100% | 확보자백: [])

박철수(냉정함): 그런 질문에는 대답할 생각이 없네.

 (남은 질문: 9 | 멘탈: 100% | 확보자백: [])

박철수(불안하지만 최대한 버티려 함): 그런 질문에는 대답할 생각이 없네.

 (남은 질문: 8 | 멘탈: 100% | 확보자백: [])

박철수(긴장됨): 대답을 거부하겠다.

 (남은 질문: 7 | 멘탈: 100% | 확보자백: [])

박철수(불안): 내 이름은 박철수야. 그런데 그게 무슨 상관이야?

 (남은 질문: 6 | 멘탈: 90% | 확보자백: [])

박철수(긴장): 그날 밤 11시 30분? 나는 집에 있었어. 그 시간에 편의점에 갔다는 영수증이 있다고? 그건 아니야.

 (남은 질문: 5 | 멘탈: 65% | 확보자백: [])

박철수(방어적): 그런 일은 전혀 없어. 그런 뻔한 질문에 답할 이유가 없네.

 (남은 질문: 4 | 멘탈: 65% | 확보자백: [])

박철수(불안): 그런 일은 없었지만, 왜 자꾸 그런 얘기를 꺼내는 거야? 난 그런 적 없다고.

 (남은 질문: 3 | 멘탈: 50% | 확보자백: [])

박철수(심리적으로 무너짐): 그... 그 증거를 어떻게 알았지? 사실 그날 밤 나는 상가에 있었어.

 (남은 질문: 2 | 멘탈: 20% | 확보자백: ['범행 시각에 상가 내부에 있었다는 CCTV 증거'])

박철수(심리적으로 크게 흔들림): 그... 그 마스터키, 사실 내 가방 안에 숨겨두고 있었어. 어떻게 알았지?

--- 취조 종료 ---
💥 승리! 박철수의 멘탈이 붕괴되어 모든 범행을 실토했습니다.
